# 03_01 — Rule-based Data Cleaning

## Mục tiêu

Notebook này tạo bản dữ liệu *interim* sạch, truy vết được từ một nguồn RAW Anh–Việt đã qua Phase 02. RAW ở `data/raw/` chỉ được đọc, không bị sửa.

Các thao tác tự động gồm reject cặp thiếu/không phải text/ký tự lỗi, chuẩn hoá Unicode và whitespace, rồi loại exact duplicate sau chuẩn hoá. Các cờ noise, URL, placeholder, length ratio, alignment và language từ audit **được mang theo trong interim artifact** để Phase 04 quyết định theo ngữ cảnh miền IT; chúng không bị tự động loại chỉ vì một heuristic.

> Chạy `Restart Kernel → Run All` riêng cho từng nguồn. Đổi duy nhất `SOURCE_SHORT_NAME` ở phần cấu hình.

In [1]:
import hashlib
import json
import os
import re
import sys
import unicodedata
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

print('Python:', sys.version)
print('pandas:', pd.__version__)
print('Working directory:', os.getcwd())

Python: 3.14.6 | packaged by Anaconda, Inc. | (main, Jul  9 2026, 14:29:05) [MSC v.1942 64 bit (AMD64)]
pandas: 3.0.5
Working directory: C:\Users\ADMIN\ENVI-IT-MT\notebooks\03_data_cleaning


In [2]:
CURRENT_DIR = Path.cwd().resolve()


def find_project_root(start_path: Path) -> Path:
    """Tìm project root từ thư mục notebook hiện tại."""
    for path in [start_path, *start_path.parents]:
        if (path / 'data').is_dir() and (path / 'notebooks').is_dir():
            return path
    raise FileNotFoundError(
        'Không tìm thấy project root chứa data/ và notebooks/. '
        'Hãy mở notebook bên trong repository ENVI-IT-MT.'
    )


PROJECT_ROOT = find_project_root(CURRENT_DIR)
print('Project root:', PROJECT_ROOT)

Project root: C:\Users\ADMIN\ENVI-IT-MT


## 1. Cấu hình nguồn và policy

Policy được version hoá trong notebook để manifest tái lập được chính xác. Thay đổi rule phải tăng `CLEANING_RULES_VERSION` và được review trước khi chạy lại.

In [3]:
SOURCE_SHORT_NAME = 'kde4'  # đổi để cleaning nguồn khác

SUPPORTED_SOURCES = [
    'envitech_reasoning',
    'tech_viet_translation',
    'gnome',
    'ubuntu',
    'kde4',
]

TEXT_COLUMN_MAPPING = {
    'envitech_reasoning': {'en': 'en', 'vi': 'vi'},
    'tech_viet_translation': {'en': 'instruction', 'vi': 'output'},
    'gnome': {'en': 'en', 'vi': 'vi'},
    'ubuntu': {'en': 'en', 'vi': 'vi'},
    'kde4': {'en': 'en', 'vi': 'vi'},
}

CLEANING_RULES_VERSION = '1.1.0'
CLEANING_POLICY = {
    'unicode_normalization': 'NFC',
    'whitespace_normalization': 'collapse_all_to_single_space',
    'reject_blank_or_null': True,
    'reject_non_string_text': True,
    'reject_replacement_or_control_character': True,
    'exact_deduplication_key': 'normalized_en + normalized_vi',
    'exact_deduplication_keep': 'first_raw_row_index',
    'audit_flag_handling': 'propagate_to_interim_for_phase_04_review',
    'auto_remove_noise_or_alignment_flags': False,
}

assert SOURCE_SHORT_NAME in SUPPORTED_SOURCES, (
    f'SOURCE_SHORT_NAME phải thuộc: {SUPPORTED_SOURCES}'
)

RAW_DIR = PROJECT_ROOT / 'data' / 'raw' / SOURCE_SHORT_NAME
AUDIT_DIR = PROJECT_ROOT / 'data' / 'audit' / SOURCE_SHORT_NAME
INTERIM_DIR = PROJECT_ROOT / 'data' / 'interim' / SOURCE_SHORT_NAME
RAW_PARQUET_PATH = RAW_DIR / f'{SOURCE_SHORT_NAME}_raw.parquet'
METADATA_PATH = RAW_DIR / 'metadata.json'
AUDIT_REPORT_PATH = AUDIT_DIR / 'audit_report.json'
REVIEW_CANDIDATES_PATH = AUDIT_DIR / 'review_candidates.csv'

INTERIM_DIR.mkdir(parents=True, exist_ok=True)
print('Source short name:', SOURCE_SHORT_NAME)
print('RAW Parquet:', RAW_PARQUET_PATH)
print('Interim directory:', INTERIM_DIR)

Source short name: kde4
RAW Parquet: C:\Users\ADMIN\ENVI-IT-MT\data\raw\kde4\kde4_raw.parquet
Interim directory: C:\Users\ADMIN\ENVI-IT-MT\data\interim\kde4


## 2. Đọc input bất biến

Cần có cả RAW Parquet, metadata Phase 01 và audit report Phase 02. Nếu audit chưa hoàn thành, notebook dừng để tránh làm sạch dữ liệu chưa được kiểm tra.

In [4]:
for required_path in [RAW_PARQUET_PATH, METADATA_PATH, AUDIT_REPORT_PATH, REVIEW_CANDIDATES_PATH]:
    assert required_path.is_file(), f'Không tìm thấy input bắt buộc: {required_path}'

raw_df = pd.read_parquet(RAW_PARQUET_PATH)
source_columns = TEXT_COLUMN_MAPPING[SOURCE_SHORT_NAME]
missing_columns = sorted(set(source_columns.values()) - set(raw_df.columns))
assert not missing_columns, f'Thiếu cột text: {missing_columns}'

with open(METADATA_PATH, 'r', encoding='utf-8') as file:
    source_metadata = json.load(file)
with open(AUDIT_REPORT_PATH, 'r', encoding='utf-8') as file:
    audit_report = json.load(file)

assert audit_report['source_short_name'] == SOURCE_SHORT_NAME
assert audit_report.get('audit_schema_version') == '1.1', (
    'Phase 03 yêu cầu audit report schema 1.1. Hãy chạy lại Phase 02 trước.'
)
assert audit_report['decision'].get('ready_for_cleaning') is True, (
    'Audit chưa cho phép cleaning. Hãy xử lý gate Phase 02 trước.'
)
assert int(audit_report['schema']['row_count']) == len(raw_df), (
    'Số hàng RAW hiện tại khác audit report. Hãy audit lại trước khi cleaning.'
)

print('Loaded RAW shape:', raw_df.shape)
audit_review_candidates = pd.read_csv(REVIEW_CANDIDATES_PATH)
required_review_columns = {'raw_row_index', 'audit_flags'}
assert required_review_columns.issubset(audit_review_candidates.columns), (
    f'Review candidates thiếu cột: {sorted(required_review_columns - set(audit_review_candidates.columns))}'
)
assert audit_review_candidates['raw_row_index'].is_unique, 'raw_row_index bị lặp trong review candidates.'

print('Audit decision:', audit_report['decision']['decision'])
print('Audit candidates carried to Phase 04:', len(audit_review_candidates))

Loaded RAW shape: (42782, 2)
Audit decision: pass_to_cleaning_with_flags
Audit candidates carried to Phase 04: 8968


In [5]:
def sha256_file(file_path: Path, chunk_size: int = 1024 * 1024) -> str:
    """Tính SHA-256 theo từng khối, không nạp toàn bộ file vào RAM."""
    digest = hashlib.sha256()
    with open(file_path, 'rb') as file:
        for chunk in iter(lambda: file.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()


raw_file_integrity = {
    'path': str(RAW_PARQUET_PATH.relative_to(PROJECT_ROOT)),
    'bytes': int(RAW_PARQUET_PATH.stat().st_size),
    'sha256': sha256_file(RAW_PARQUET_PATH),
}

assert raw_file_integrity['sha256'] == audit_report['raw_file_integrity']['sha256'], (
    'SHA-256 RAW khác với file đã audit. Hãy chạy lại Phase 02 trước.'
)

audit_report_integrity = {
    'path': str(AUDIT_REPORT_PATH.relative_to(PROJECT_ROOT)),
    'bytes': int(AUDIT_REPORT_PATH.stat().st_size),
    'sha256': sha256_file(AUDIT_REPORT_PATH),
}
raw_file_integrity, audit_report_integrity

({'path': 'data\\raw\\kde4\\kde4_raw.parquet',
  'bytes': 1551207,
  'sha256': '0171299fe73223aea450e7667d388940b76da3013c6e4c20c91ba5e7647fc521'},
 {'path': 'data\\audit\\kde4\\audit_report.json',
  'bytes': 4468,
  'sha256': '13f9ee65bf07323fc53d55c1d7218543d24ff100769ef4abe1da3cf82cfdede7'})

## 3. Hàm cleaning xác định

Các hàm dưới đây thuần (deterministic): cùng input và policy luôn cho cùng output. `raw_text` được giữ nguyên trong artifact; bản normalized chỉ dùng cho output và deduplication.

In [6]:
CONTROL_CHAR_RE = re.compile(r'[\x00-\x08\x0B\x0C\x0E-\x1F\x7F-\x9F]')
WHITESPACE_RE = re.compile(r'\s+')
REPLACEMENT_CHAR = '\ufffd'


def is_missing_or_blank(value: object) -> bool:
    return pd.isna(value) or (isinstance(value, str) and not value.strip())


def is_text_string(value: object) -> bool:
    return isinstance(value, str)


def has_encoding_issue(value: object) -> bool:
    if not isinstance(value, str):
        return False
    return REPLACEMENT_CHAR in value or bool(CONTROL_CHAR_RE.search(value))


def normalize_text(value: str) -> str:
    value = unicodedata.normalize('NFC', value)
    return WHITESPACE_RE.sub(' ', value).strip()


def pair_sha256(en_text: str, vi_text: str) -> str:
    # JSON array tránh collision có thể xảy ra nếu text chứa delimiter.
    payload = json.dumps([en_text, vi_text], ensure_ascii=False, separators=(',', ':')).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()

## 4. Tạo bảng working và cờ reject

Mỗi hàng RAW nhận đúng một hành động. Lý do được ưu tiên theo thứ tự: blank/null → non-string → encoding → exact duplicate. Thứ tự này bảo đảm count reject là loại trừ lẫn nhau.

In [7]:
cleaning_df = pd.DataFrame({
    'source_short_name': SOURCE_SHORT_NAME,
    'raw_row_index': raw_df.index.astype('int64'),
    'en_raw': raw_df[source_columns['en']],
    'vi_raw': raw_df[source_columns['vi']],
})

audit_flags_by_row = audit_review_candidates[['raw_row_index', 'audit_flags']].copy()
audit_flags_by_row['raw_row_index'] = audit_flags_by_row['raw_row_index'].astype('int64')
cleaning_df = cleaning_df.merge(audit_flags_by_row, on='raw_row_index', how='left', validate='one_to_one')
cleaning_df['audit_flags'] = cleaning_df['audit_flags'].fillna('').astype('string')
cleaning_df['requires_phase_04_review'] = cleaning_df['audit_flags'].ne('')

cleaning_df['blank_or_null'] = (
    cleaning_df['en_raw'].map(is_missing_or_blank)
    | cleaning_df['vi_raw'].map(is_missing_or_blank)
)
cleaning_df['non_string_text'] = (
    ~cleaning_df['en_raw'].map(is_text_string)
    | ~cleaning_df['vi_raw'].map(is_text_string)
) & ~cleaning_df['blank_or_null']
cleaning_df['encoding_issue'] = (
    cleaning_df['en_raw'].map(has_encoding_issue)
    | cleaning_df['vi_raw'].map(has_encoding_issue)
) & ~cleaning_df['blank_or_null'] & ~cleaning_df['non_string_text']

valid_text_mask = ~(
    cleaning_df['blank_or_null']
    | cleaning_df['non_string_text']
    | cleaning_df['encoding_issue']
)
cleaning_df.loc[valid_text_mask, 'en_clean'] = cleaning_df.loc[valid_text_mask, 'en_raw'].map(normalize_text)
cleaning_df.loc[valid_text_mask, 'vi_clean'] = cleaning_df.loc[valid_text_mask, 'vi_raw'].map(normalize_text)

assert cleaning_df.loc[valid_text_mask, ['en_clean', 'vi_clean']].notna().all().all()
cleaning_df.head()

,source_short_name,raw_row_index,en_raw,vi_raw,audit_flags,requires_phase_04_review,blank_or_null,non_string_text,encoding_issue,en_clean,vi_clean
0,kde4,0,Add Feed to Akregator,Thêm nguồn tin cho Akregator,,False,False,False,False,Add Feed to Akregator,Thêm nguồn tin cho Akregator
1,kde4,1,Add Feeds to Akregator,Thêm các nguồn tin cho Akregator,,False,False,False,False,Add Feeds to Akregator,Thêm các nguồn tin cho Akregator
2,kde4,2,Add All Found Feeds to Akregator,Thêm mọi nguồn tin cho Akregator,,False,False,False,False,Add All Found Feeds to Akregator,Thêm mọi nguồn tin cho Akregator
3,kde4,3,Subscribe to site updates (using news feed),Theo dõi chỗ Mạng này tìm bản cập nhật (dùng n...,,False,False,False,False,Subscribe to site updates (using news feed),Theo dõi chỗ Mạng này tìm bản cập nhật (dùng n...
4,kde4,4,Imported Feeds,Nguồn tin đã nhập,,False,False,False,False,Imported Feeds,Nguồn tin đã nhập


In [8]:
# Chỉ deduplicate các cặp đã qua validation cơ bản; giữ hàng RAW đầu tiên.
cleaning_df['exact_duplicate'] = False
dedup_candidate_mask = valid_text_mask
cleaning_df.loc[dedup_candidate_mask, 'exact_duplicate'] = cleaning_df.loc[
    dedup_candidate_mask
].duplicated(subset=['en_clean', 'vi_clean'], keep='first').to_numpy()

cleaning_df['pair_sha256'] = pd.NA
cleaning_df.loc[dedup_candidate_mask, 'pair_sha256'] = [
    pair_sha256(en_text, vi_text)
    for en_text, vi_text in cleaning_df.loc[dedup_candidate_mask, ['en_clean', 'vi_clean']].itertuples(index=False, name=None)
]

cleaning_df['cleaning_action'] = 'keep'
cleaning_df['rejection_reason'] = pd.NA
for reason in ['blank_or_null', 'non_string_text', 'encoding_issue', 'exact_duplicate']:
    reason_mask = cleaning_df[reason] & cleaning_df['rejection_reason'].isna()
    cleaning_df.loc[reason_mask, 'cleaning_action'] = 'reject'
    cleaning_df.loc[reason_mask, 'rejection_reason'] = reason

assert cleaning_df['rejection_reason'].notna().eq(cleaning_df['cleaning_action'].eq('reject')).all()
cleaning_df['cleaning_action'].value_counts()

cleaning_action
keep      39888
reject     2894
Name: count, dtype: int64

## 5. Tạo artifact keep/reject

Hai artifact giữ `raw_row_index`, nội dung ban đầu và nội dung cleaned để tái lập hoặc kiểm tra một quyết định bất kỳ.

In [9]:
OUTPUT_COLUMNS = [
    'source_short_name', 'raw_row_index', 'en_raw', 'vi_raw',
    'en_clean', 'vi_clean', 'pair_sha256', 'audit_flags', 'requires_phase_04_review', 'cleaning_action',
    'rejection_reason',
]

cleaned_pairs = cleaning_df.loc[
    cleaning_df['cleaning_action'].eq('keep'), OUTPUT_COLUMNS
].sort_values('raw_row_index').reset_index(drop=True)
rejected_pairs = cleaning_df.loc[
    cleaning_df['cleaning_action'].eq('reject'), OUTPUT_COLUMNS
].sort_values('raw_row_index').reset_index(drop=True)

print('Kept pairs:', len(cleaned_pairs))
print('Rejected pairs:', len(rejected_pairs))
display(rejected_pairs['rejection_reason'].value_counts(dropna=False).to_frame('count'))

Kept pairs: 39888
Rejected pairs: 2894


,count
rejection_reason,
exact_duplicate,2888
encoding_issue,6


## 6. Báo cáo cleaning

Các count reject loại trừ lẫn nhau; `raw_flag_counts` giúp so sánh với audit ngay cả khi một hàng có nhiều cờ.

In [10]:
raw_flag_counts = {
    reason: int(cleaning_df[reason].sum())
    for reason in ['blank_or_null', 'non_string_text', 'encoding_issue', 'exact_duplicate']
}
rejection_counts = {
    reason: int(rejected_pairs['rejection_reason'].eq(reason).sum())
    for reason in ['blank_or_null', 'non_string_text', 'encoding_issue', 'exact_duplicate']
}

cleaning_summary = pd.DataFrame([
    {'source_short_name': SOURCE_SHORT_NAME, 'raw_rows': len(raw_df),
     'kept_rows': len(cleaned_pairs), 'rejected_rows': len(rejected_pairs),
     **{f'rejected_{name}': count for name, count in rejection_counts.items()}}
])
display(cleaning_summary)

,source_short_name,raw_rows,kept_rows,rejected_rows,rejected_blank_or_null,rejected_non_string_text,rejected_encoding_issue,rejected_exact_duplicate
0,kde4,42782,39888,2894,0,0,6,2888


In [11]:
def json_ready(value):
    """Chuyển pandas/Path/NaN sang kiểu JSON an toàn."""
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    if isinstance(value, Path):
        return str(value)
    if pd.isna(value):
        return None
    if hasattr(value, 'item'):
        return value.item()
    return value


run_timestamp_utc = datetime.now(timezone.utc).isoformat()
cleaning_report = {
    'cleaning_schema_version': '1.1',
    'cleaning_timestamp_utc': run_timestamp_utc,
    'source_short_name': SOURCE_SHORT_NAME,
    'raw_file_integrity': raw_file_integrity,
    'audit_report_path': str(AUDIT_REPORT_PATH.relative_to(PROJECT_ROOT)),
    'audit_report_integrity': audit_report_integrity,
    'audit_report_timestamp_utc': audit_report['audit_timestamp_utc'],
    'audit_decision': {
        'decision': audit_report['decision']['decision'],
        'ready_for_cleaning': audit_report['decision']['ready_for_cleaning'],
        'approved_for_dataset_building': audit_report['decision']['approved_for_dataset_building'],
    },
    'rules_version': CLEANING_RULES_VERSION,
    'policy': CLEANING_POLICY,
    'input_rows': int(len(raw_df)),
    'kept_rows': int(len(cleaned_pairs)),
    'rejected_rows': int(len(rejected_pairs)),
    'raw_flag_counts_non_exclusive': raw_flag_counts,
    'rejection_counts_exclusive': rejection_counts,
    'rows_requiring_phase_04_review': int(cleaned_pairs['requires_phase_04_review'].sum()),
}
cleaning_report

{'cleaning_schema_version': '1.1',
 'cleaning_timestamp_utc': '2026-08-27T08:30:21.220411+00:00',
 'source_short_name': 'kde4',
 'raw_file_integrity': {'path': 'data\\raw\\kde4\\kde4_raw.parquet',
  'bytes': 1551207,
  'sha256': '0171299fe73223aea450e7667d388940b76da3013c6e4c20c91ba5e7647fc521'},
 'audit_report_path': 'data\\audit\\kde4\\audit_report.json',
 'audit_report_integrity': {'path': 'data\\audit\\kde4\\audit_report.json',
  'bytes': 4468,
  'sha256': '13f9ee65bf07323fc53d55c1d7218543d24ff100769ef4abe1da3cf82cfdede7'},
 'audit_report_timestamp_utc': '2026-08-27T08:15:32.574023+00:00',
 'audit_decision': {'decision': 'pass_to_cleaning_with_flags',
  'ready_for_cleaning': True,
  'approved_for_dataset_building': False},
 'rules_version': '1.1.0',
 'policy': {'unicode_normalization': 'NFC',
  'whitespace_normalization': 'collapse_all_to_single_space',
  'reject_blank_or_null': True,
  'reject_non_string_text': True,
  'reject_replacement_or_control_character': True,
  'exact_dedu

## 7. Lưu artifact và manifest

Parquet là artifact chính để giữ kiểu dữ liệu; JSONL hỗ trợ inspect/trao đổi. Manifest chứa checksum output để phát hiện thay đổi ngoài ý muốn.

In [12]:
CLEANED_PARQUET_PATH = INTERIM_DIR / 'cleaned_pairs.parquet'
CLEANED_JSONL_PATH = INTERIM_DIR / 'cleaned_pairs.jsonl'
REJECTED_PARQUET_PATH = INTERIM_DIR / 'rejected_pairs.parquet'
CLEANING_REPORT_PATH = INTERIM_DIR / 'cleaning_report.json'
CLEANING_MANIFEST_PATH = INTERIM_DIR / 'cleaning_manifest.json'

cleaned_pairs.to_parquet(CLEANED_PARQUET_PATH, index=False)
cleaned_pairs.to_json(CLEANED_JSONL_PATH, orient='records', lines=True, force_ascii=False)
rejected_pairs.to_parquet(REJECTED_PARQUET_PATH, index=False)
with open(CLEANING_REPORT_PATH, 'w', encoding='utf-8') as file:
    json.dump(json_ready(cleaning_report), file, ensure_ascii=False, indent=2, allow_nan=False)

print('Saved:', CLEANED_PARQUET_PATH)
print('Saved:', CLEANED_JSONL_PATH)
print('Saved:', REJECTED_PARQUET_PATH)
print('Saved:', CLEANING_REPORT_PATH)

Saved: C:\Users\ADMIN\ENVI-IT-MT\data\interim\kde4\cleaned_pairs.parquet
Saved: C:\Users\ADMIN\ENVI-IT-MT\data\interim\kde4\cleaned_pairs.jsonl
Saved: C:\Users\ADMIN\ENVI-IT-MT\data\interim\kde4\rejected_pairs.parquet
Saved: C:\Users\ADMIN\ENVI-IT-MT\data\interim\kde4\cleaning_report.json


In [13]:
output_files = [
    CLEANED_PARQUET_PATH, CLEANED_JSONL_PATH, REJECTED_PARQUET_PATH, CLEANING_REPORT_PATH,
]
cleaning_manifest = {
    'manifest_schema_version': '1.0',
    'created_at_utc': run_timestamp_utc,
    'source_short_name': SOURCE_SHORT_NAME,
    'rules_version': CLEANING_RULES_VERSION,
    'raw_file_sha256': raw_file_integrity['sha256'],
    'audit_report_sha256': audit_report_integrity['sha256'],
    'artifacts': [
        {
            'path': str(path.relative_to(PROJECT_ROOT)),
            'bytes': int(path.stat().st_size),
            'sha256': sha256_file(path),
        }
        for path in output_files
    ],
}
with open(CLEANING_MANIFEST_PATH, 'w', encoding='utf-8') as file:
    json.dump(json_ready(cleaning_manifest), file, ensure_ascii=False, indent=2, allow_nan=False)

print('Saved:', CLEANING_MANIFEST_PATH)

Saved: C:\Users\ADMIN\ENVI-IT-MT\data\interim\kde4\cleaning_manifest.json


## 8. Verification

Các bất biến dưới đây phải đúng trước khi công nhận output. Bất kỳ assertion nào lỗi đều phải được điều tra; không bỏ qua bằng cách sửa output thủ công.

In [14]:
# 1. Không mất hoặc nhân bản hàng RAW.
all_output_indexes = pd.concat([cleaned_pairs['raw_row_index'], rejected_pairs['raw_row_index']])
assert len(all_output_indexes) == len(raw_df)
assert all_output_indexes.nunique() == len(raw_df)
assert set(all_output_indexes) == set(raw_df.index)

# 2. Artifact cleaned không còn lỗi bị policy reject.
assert cleaned_pairs['en_clean'].map(lambda value: isinstance(value, str) and bool(value.strip())).all()
assert cleaned_pairs['vi_clean'].map(lambda value: isinstance(value, str) and bool(value.strip())).all()
assert not cleaned_pairs['en_clean'].map(has_encoding_issue).any()
assert not cleaned_pairs['vi_clean'].map(has_encoding_issue).any()

# 3. Không còn exact duplicate sau normalized cleaning.
assert not cleaned_pairs.duplicated(subset=['en_clean', 'vi_clean']).any()
assert cleaned_pairs['pair_sha256'].notna().all()
assert cleaned_pairs['audit_flags'].notna().all()
assert cleaned_pairs['requires_phase_04_review'].eq(cleaned_pairs['audit_flags'].ne('')).all()

# 4. File và manifest tồn tại; có thể đọc lại artifact chính.
expected_files = output_files + [CLEANING_MANIFEST_PATH]
assert all(path.is_file() for path in expected_files)
reloaded_cleaned_pairs = pd.read_parquet(CLEANED_PARQUET_PATH)
assert len(reloaded_cleaned_pairs) == len(cleaned_pairs)

print('Verification passed:', True)
print('Rows: raw =', len(raw_df), '| kept =', len(cleaned_pairs), '| rejected =', len(rejected_pairs))

Verification passed: True
Rows: raw = 42782 | kept = 39888 | rejected = 2894


## 9. Cross-source cleaning summary

Sau khi chạy notebook cho đủ năm nguồn, chạy cell này ở bất kỳ source nào để tổng hợp output. Cell chỉ đọc report đã có; không làm sạch lại dữ liệu.

In [15]:
cross_source_rows = []
for source_short_name in SUPPORTED_SOURCES:
    report_path = PROJECT_ROOT / 'data' / 'interim' / source_short_name / 'cleaning_report.json'
    if not report_path.is_file():
        continue
    with open(report_path, 'r', encoding='utf-8') as file:
        report = json.load(file)
    cross_source_rows.append({
        'source_short_name': source_short_name,
        'input_rows': report['input_rows'],
        'kept_rows': report['kept_rows'],
        'rejected_rows': report['rejected_rows'],
        **{f'rejected_{name}': count for name, count in report['rejection_counts_exclusive'].items()},
        'rows_requiring_phase_04_review': report.get('rows_requiring_phase_04_review'),
        'audit_report_sha256': report.get('audit_report_integrity', {}).get('sha256'),
        'rules_version': report['rules_version'],
        'cleaning_timestamp_utc': report['cleaning_timestamp_utc'],
    })

cross_source_summary = pd.DataFrame(cross_source_rows).sort_values('source_short_name').reset_index(drop=True)
CROSS_SOURCE_SUMMARY_CSV = PROJECT_ROOT / 'data' / 'interim' / 'cross_source_cleaning_summary.csv'
CROSS_SOURCE_SUMMARY_JSON = PROJECT_ROOT / 'data' / 'interim' / 'cross_source_cleaning_summary.json'
cross_source_summary.to_csv(CROSS_SOURCE_SUMMARY_CSV, index=False, encoding='utf-8-sig')
with open(CROSS_SOURCE_SUMMARY_JSON, 'w', encoding='utf-8') as file:
    json.dump(json_ready(cross_source_rows), file, ensure_ascii=False, indent=2, allow_nan=False)

print(f'Cleaned sources: {len(cross_source_summary)}/{len(SUPPORTED_SOURCES)}')
display(cross_source_summary)

Cleaned sources: 5/5


,source_short_name,input_rows,kept_rows,rejected_rows,rejected_blank_or_null,rejected_non_string_text,rejected_encoding_issue,rejected_exact_duplicate,rows_requiring_phase_04_review,audit_report_sha256,rules_version,cleaning_timestamp_utc
0,envitech_reasoning,15115,13690,1425,0,0,0,1425,1040,e2c0d2f30827f4b93d3634f499f37bdc86312be7b4d0f5...,1.1.0,2026-08-27T08:25:31.156321+00:00
1,gnome,149,148,1,0,0,0,1,16,446cfedabf7e234b1f24cf07e69c5a9e4681a07388175f...,1.1.0,2026-08-27T08:27:54.535924+00:00
2,kde4,42782,39888,2894,0,0,6,2888,6075,13f9ee65bf07323fc53d55c1d7218543d24ff100769ef4...,1.1.0,2026-08-27T08:30:21.220411+00:00
3,tech_viet_translation,100767,100458,309,0,0,0,309,6035,fdb8a91045271eeddd8d653a46b06ac2d321a9446d009b...,1.1.0,2026-08-27T08:27:21.533818+00:00
4,ubuntu,5056,5036,20,0,0,0,20,1650,ce972ac1177d22cb070bdd9deffcfb216857e4a5ab33d2...,1.1.0,2026-08-27T08:29:04.238060+00:00


# Data Cleaning Status

## Đã thực hiện

- [x] Đọc RAW và audit report schema 1.1, đối chiếu SHA-256 và gate `ready_for_cleaning`.
- [x] Reject cặp blank/null, non-string và replacement/control character.
- [x] Chuẩn hoá NFC + whitespace và loại exact duplicate có thứ tự xác định.
- [x] Mang audit flags vào interim artifact cho Phase 04; lưu keep/reject artifact, report, manifest checksum và cross-source summary.
- [x] Kiểm tra conservation of rows, chất lượng output và deduplication.

## Chưa thực hiện

- [ ] Lọc miền IT (Phase 04).
- [ ] Semantic alignment hoặc near-duplicate cleaning cần mô hình/đánh giá thủ công.
- [ ] Train/validation/test split (Phase 05).

> Không commit `data/interim/` vì đã được `.gitignore`; commit notebook và tài liệu/quy tắc.